In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("clean-data-w-spark-data-data").getOrCreate()

26/11/20 19:52:07 WARN Utils: Your hostname, DDG-1000 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/20 19:52:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 19:52:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/20 19:52:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [17]:
df18 = spark.read.format("csv").options(header=True).load("AA_DFW_2015_Departures_Short.csv.gz")

df18.toPandas().to_csv("2015-departures.csv.gz", index=False, compression="gzip")

In [8]:
departures_df.toPandas().to_csv("departures.txt.gz", index=False, compression="gzip")

In [13]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/datasets/airport-codes/main/data/airport-codes.csv")
df = df[["name", "iata_code"]]
df = df[df["iata_code"].notnull()]
df = df.rename(
    columns={
        "name": "AIRPORTNAME",
        "iata_code": "IATA"
    }
)

df.to_csv("airports_df.csv", index=False)

In [4]:
df.sample(n=7)

,ident,type,name,elevation_ft,continent,iso_country,iso_region,municipality,icao_code,iata_code,gps_code,local_code,coordinates
10944,AR-0704,small_airport,Estancia La Maciega Airport,620.0,SA,AR,AR-U,Florentino Ameghino,NaN,NaN,NaN,NaN,"-44.45854, -65.60563"
21906,DA12,small_airport,El Abiodh Sidi Cheikh Airport,2965.0,AF,DZ,DZ-32,El Abiodh Sidi Cheikh,NaN,NaN,DA12,DA12,"32.898499, 0.524694"
61008,SJCY,small_airport,Estância Santa Rita Airport,617.0,SA,BR,BR-MT,Cuiabá,NaN,NaN,SJCY,MT0166,"-15.680085, -55.95603"
71267,US-12565,closed,General Plumbing Contractors Heliport,80.0,NaN,US,US-TX,Houston,NaN,NaN,NaN,NaN,"29.897, -95.395167"
30249,GE-0014,heliport,Isani-Samgori Heliport,1939.0,AS,GE,GE-TB,Isani-Samgori,NaN,NaN,NaN,NaN,"41.651019, 44.870111"
50739,NL-0024,small_airport,Middenmeer Airfield,-14.0,EU,NL,NL-NH,Middenmeer,EHMM,NaN,EHMM,NaN,"52.814876, 5.021245"
36085,JP-1804,closed,Iwakuni Airfield,NaN,AS,JP,JP-35,Iwakuni,NaN,NaN,NaN,NaN,"34.154, 132.23539"


In [18]:
import os, gzip
import xml.etree.ElementTree as ET

ANNOT_DIR = "Annotation"
OUTPUT = "annotations.csv.gz"

def flatten(path):
    root = ET.parse(path).getroot()
    folder   = root.findtext("folder", "")
    filename = root.findtext("filename", "")
    size     = root.find("size")
    width    = size.findtext("width", "")  if size is not None else ""
    height   = size.findtext("height", "") if size is not None else ""

    dogs = []
    for obj in root.findall("object"):
        name = obj.findtext("name", "")
        box  = obj.find("bndbox")
        coords = [box.findtext(k, "") for k in ("xmin", "ymin", "xmax", "ymax")]
        dogs.append(",".join([name] + coords))   # breed,xmin,ymin,xmax,ymax

    # folder \t filename \t width \t height \t dog1 \t dog2 ...
    return "\t".join([folder, filename, width, height] + dogs)

rows = []
for dirpath, _, files in os.walk(ANNOT_DIR):
    for fn in files:
        try:
            rows.append(flatten(os.path.join(dirpath, fn)))
        except ET.ParseError:
            continue  # skip anything that isn't valid XML

with gzip.open(OUTPUT, "wt") as f:
    f.write("\n".join(rows) + "\n")

print(f"Wrote {len(rows)} rows -> {OUTPUT}")

Wrote 20580 rows -> annotations.csv.gz


In [3]:
import pandas as pd

folders = ['02085620', '02085782', '02085936', '02086079', '02086240', '02086646', '02086910', '02087046', '02087394', '02088094', '02088238', '02088364', '02088466', '02088632', '02089078', '02089867', '02090379', '02090622', '02090721', '02091032', '02091134', '02091244', '02091467', '02091831', '02092002', '02092339', '02093256', '02093428', '02093647', '02093754', '02093859', '02093991', '02094114', '02094258', '02094433', '02095314', '02095570', '02095889', '02096051', '02096177', '02096294', '02096437', '02096585', '02097047', '02097130', '02097209', '02097298', '02097474', '02097658', '02098105', '02098286', '02098413', '02099267', '02099429', '02099601', '02099712', '02099849', '02100160', '02100236', '02100583', '02100735', '02100877', '02101006', '02101388', '02101556', '02102040', '02102177', '02102318', '02102480', '02102778', '02102973', '02104029', '02104365', '02105056', '02105162', '02105251', '02105412', '02105505', '02105641', '02106030', '02106166', '02106371', '02106382', '02106550', '02106662', '02106706', '02106830', '02107056', '02107142', '02107312', '02107371', '02107574', '02107683', '02107908', '02108000', '02108089', '02108151', '02108422', '02108551', '02108578', '02109047', '02109525', '02109961', '02110063', '02110185', '02110231', '02110627', '02110806', '02110958', '02111129', '02111277', '02111451', '02111500', '02111889', '02112018', '02112137', '02112350', '02112367', '02112706', '02112762', '02113023', '02113186', '02113624', '02113712', '02113799', '02113879', '02113978', '02114249', '02114866', '02115641', '02115913', '02116738', '02116983', '02117158', '02120402', '02120539', '02120858', '02123243', '02123757', '02124007', '02124701', '02126513', '02126980', '02127354', '02127818', '02129095', '02129135', '02129214', '02129463', '02129790', '02129912', '02130837', '02131532', '02133619', '02133963', '02134643', '02135177', '02136377', '02136426', '02137175', '02138431', '02140630', '02142438', '02142773', '02143015', '02143756', '02144179', '02144878', '02146062', '02147142', '02148654', '02149218', '02151614', '02153916', '02153947', '02154048', '02154389', '02155648', '02156171', '02156221', '02156958', '02157469', '02159250', '02160392', '02160989', '02165095', '02165474', '02167770', '02167944', '02168743', '02168949', '02168957', '02173211', '02173325', '02173731', '02173813', '02175169', '02176285', '02176979', '02178704', '02179033', '02181725', '02182158', '02182938', '02184195', '02184488', '02186483', '02186577', '02187790', '02188302', '02188401', '02188727', '02190379', '02190582', '02191289', '02191617', '02192205', '02192768', '02196478', '02196511', '02196678', '02196833', '02197270', '02197373', '02197503', '02197508', '02197882', '02198170', '02199899', '02200232', '02202104', '02203580', '02204471', '02205121', '02205252', '02205898', '02206272', '02206349', '02208295', '02208652', '02210266', '02210991', '02211526', '02212090', '02213092', '02213276', '02213751', '02214012', '02214120', '02215138', '02215626', '02216022', '02216327', '02216928', '02217665', '02218819', '02219333', '02219584', '02221269', '02221407', '02221420', '02222280', '02222333', '02222953', '02224195', '02224935', '02227816', '02227916', '02229328', '02229538', '02232573', '02234190', '02234700', '02235858', '02237062', '02238079', '02239027', '02242135', '02242647', '02245529', '02247460', '02248316', '02248634', '02249073', '02249724', '02253426', '02253920', '02254181', '02254309', '02255235', '02255277', '02255360', '02255862', '02256594', '02257098', '02257953', '02258337', '02258812', '02260244', '02262893', '02263578', '02267337', '02267791', '02268288', '02268482', '02269977', '02270391', '02270667', '02270994', '02271086', '02271511', '02273630', '02273807', '02274980', '02279834', '02280107', '02283043', '02283049', '02283402', '02285453', '02286085', '02287595', '02288634', '02289390', '02289426', '02290405', '02291212', '02292010', '02293170', '02293649', '02293967', '02294346', '02295287', '02295548', '02296632', '02296976', '02297302', '02297746', '02297969', '02298155', '02298393', '02300988', '02301050', '02301077', '02302114', '02302770', '02303812', '02305476', '02307420', '02307488', '02307680', '02308167', '02309077', '02312663', '02312706', '02312883', '02314020', '02314021', '02314393', '02314431', '02315286', '02315808', '02316140', '02316295', '02316725', '02316750', '02316948', '02317649', '02318221', '02318471', '02323261', '02323381', '02324938', '02324963', '02325040', '02325532', '02326410', '02326752', '02331447', '02332513', '02334040', '02334643', '02334910', '02335549', '02337505', '02338469', '02339341', '02340410', '02342025', '02342557', '02343105', '02343480', '02343762', '02343949', '02344860', '02346778', '02346879', '02347219', '02347298', '02348361', '02348748', '02348749', '02349177', '02349520', '02350525', '02350798', '02351010', '02351350', '02352327', '02352387', '02352532', '02353969', '02356240', '02356649', '02356775', '02357859', '02358063', '02358275', '02359333', '02359428', '02359860', '02360076', '02361162', '02362732', '02363754', '02364325', '02364926', '02365052', '02366261', '02368172', '02368477', '02369976', '02370493', '02371544', '02371666', '02372631', '02375229', '02376088', '02378229', '02378538', '02380295', '02380302', '02380501', '02381340', '02381428', '02383557', '02383958', '02384833', '02387732', '02387825', '02388514', '02389509', '02390200', '02390881', '02391234', '02393203', '02396191', '02397415', '02397667', '02398676', '02398827', '02398955', '02399972', '02400100', '02400455', '02401197', '02401298', '02403259', '02404362', '02404795', '02404926', '02406019', '02407407', '02407523', '02407891', '02408828', '02411180', '02413827', '02414552', '02414702', '02418468', '02418476', '02419011', '02419697', '02419939', '02421335', '02424400', '02425606', '02426108', '02426484', '02428153', '02429489', '02429831', '02429874', '02430003', '02432982', '02434869', '02435891', '02436451', '02436911', '02436935', '02437102', '02437591', '02437593', '02438178', '02439793', '02440756', '02440821', '02441159', '02441895', '02442336', '02442958', '02443054', '02443194', '02445171', '02445610', '02445654', '02446438', '02448103', '02449428', '02450005', '02450563', '02450766', '02450794', '02451531', '02452210', '02453882', '02454905', '02458013', '02459642', '02459857', '02460067', '02460651', '02461921', '02465927', '02466479', '02467194', '02467626', '02472947', '02473142', '02473854', '02474135', '02475627', '02476464', '02479425', '02480317', '02480764', '02480900', '02482207', '02483003', '02484348', '02484657', '02486239', '02486878', '02489700', '02491196', '02493776', '02494915', '02495786', '02498674', '02498844', '02499321', '02499925', '02501269', '02501774', '02502915', '02503705', '02504936', '02505774', '02505910', '02507519', '02509063', '02511784', '02511993', '02513722', '02515972', '02518641', '02520029', '02520624', '02520976', '02522217', '02522773', '02523614', '02524305', '02525173', '02525746', '02526109', '02526414', '02527185', '02527328', '02528250', '02530969', '02531045', '02532160', '02532332', '02532773', '02534720', '02535915', '02537023', '02537037', '02537765', '02537929', '02538799', '02538821', '02538928', '02541366', '02542517', '02543698', '02544312', '02545873', '02546430', '02547135', '02547485', '02549653', '02552672', '02552686', '02553953', '02554163', '02555193', '02558147', '02558493', '02559773', '02559918', '02560816', '02562476', '02562836', '02562931', '02564141', '02564727', '02565141', '02567848', '02567900', '02567960', '02568884', '02573972', '02574909', '02575709', '02575755', '02576262', '02577526', '02578151', '02578743', '02578875', '02579315', '02580696', '02580979', '02581620', '02583095', '02584103', '02585418', '02586004', '02586066', '02586359', '02586460', '02589208', '02589831', '02590234', '02590246', '02590810', '02591353', '02592468', '02593080', '02594570', '02594728', '02595006', '02595928', '02596248', '02597542', '02597967', '02598132', '02598158', '02599137', '02599759', '02601129', '02601858', '02602620', '02608636', '02609765', '02611313', '02612403', '02613273', '02613681', '02614288', '02615191', '02616789', '02616911', '02617038', '02617129', '02618347', '02621044', '02621898', '02625331', '02626715', '02626879', '02627100', '02628132', '02628222', '02628919', '02633187', '02633310', '02633549', '02634281', '02635009', '02635593', '02636640', '02638739', '02640307', '02642905', '02644948', '02645354', '02646040', '02646600', '02649638', '02649741', '02649986', '02653377', '02654109', '02654661', '02654696', '02655023', '02656468', '02656869', '02657500', '02657631', '02658000', '02658236', '02659007', '02659717', '02660649', '02661724', '02662307', '02663655', '02665281', '02665429', '02665642', '02666001', '02666068', '02667558', '02668579', '02669362', '02669442', '02669853', '02672361', '02672630', '02673179', '02675647', '02676480', '02676829', '02678857', '02680877', '02681896', '02683942', '02685037', '02685370', '02686354', '02687786', '02690630', '02691629', '02692321', '02693257', '02696342', '02696876', '02697079', '02697915', '02698153', '02698939', '02699496', '02699692', '02700076', '02700641', '02701592', '02702096', '02702960', '02703205', '02703725', '02705210', '02705478', '02705727', '02706165', '02706388', '02706438', '02707637', '02708927', '02709174', '02709435', '02709989', '02710263', '02712140', '02712319', '02712560', '02714651', '02715113', '02716060', '02717626', '02718296', '02718657', '02719088', '02723988', '02724613', '02726918', '02727317', '02732029', '02732189', '02732391', '02733527', '02733733', '02735891', '02738359', '02738701', '02738881', '02739521', '02739637', '02739817', '02739971', '02740171', '02740496', '02741030', '02741267', '02741767', '02741920', '02742724', '02742904', '02743916', '02744051', '02745776', '02748014', '02748225', '02748389', '02748612', '02748689', '02749821', '02750291', '02750961', '02750999', '02751088', '02751492', '02753729', '02754259', '02756067', '02756750', '02758270', '02759463', '02762228', '02763398', '02763460', '02766597', '02767032', '02768926', '02769456', '02771075', '02771175', '02773132', '02774227', '02776610', '02776649', '02777745', '02778329', '02778355', '02780947', '02781325', '02781610', '02784680', '02785383', '02786254', '02786991', '02787012', '02788031', '02788283', '02788354', '02790839', '02792174', '02792212', '02793222', '02794110', '02794791', '02796350', '02798293', '02799025', '02799191', '02800490', '02800740', '02801614', '02801869', '02802129', '02802901', '02802979', '02803608', '02808286', '02809418', '02809586', '02810031', '02810787', '02811618', '02813107', '02813918', '02815126', '02816400', '02818496', '02819961', '02820513', '02821227', '02821523', '02822168', '02822460', '02822497', '02822853', '02824560', '02826314', '02826413', '02827644', '02827837', '02829211', '02829668', '02829699', '02829961', '02832654', '02832690', '02833175', '02833657', '02835049', '02835482', '02835501', '02835532', '02835556', '02835897', '02836729', '02836913', '02837059', '02837349', '02837364', '02837381', '02838318', '02838828', '02839370', '02839690', '02839977', '02842769', '02843192', '02844180', '02845079', '02845122', '02845788', '02845872', '02848210', '02848814', '02848871', '02851781', '02852564', '02855753', '02855935', '02858694', '02858820', '02859367', '02859642', '02859704', '02860145', '02860330', '02860483', '02860983', '02862053', '02862069', '02863668', '02866889', '02868655', '02868725', '02868808', '02869288', '02871884', '02872850', '02873616', '02874348', '02878005', '02878348', '02879018', '02882664', '02885293', '02888119', '02889596', '02890686', '02890787', '02891592', '02892282', '02895260', '02896737', '02897220', '02898034', '02899208', '02899508', '02899673', '02899769', '02900709', '02900877', '02901950', '02903954', '02904123', '02904376', '02904857', '02906015', '02907411', '02907502', '02907870', '02907992', '02908179', '02908557', '02908709', '02910459', '02910486', '02912693', '02913105', '02913451', '02913955', '02915579', '02915694', '02915850', '02916837', '02917335', '02920304', '02921596', '02922028', '02922296', '02922892', '02924237', '02925556', '02925638', '02926457', '02926548', '02927769', '02928383', '02929322', '02930126', '02930490', '02930904', '02933763', '02934990', '02936285', '02937697', '02938064', '02938152', '02938891', '02938994', '02939107', '02939748', '02943031', '02943347', '02944042', '02945419', '02945618', '02946092', '02946523', '02947472', '02947877', '02948479', '02951150', '02951503', '02952330', '02952336', '02952367', '02952957', '02953561', '02954613', '02955062', '02955176', '02956177', '02958147', '02958962', '02960595', '02961669', '02964365', '02966989', '02968394', '02968546', '02971140', '02971526', '02972901', '02973035', '02974516', '02976732', '02978813', '02980114', '02980444', '02980644', '02981901', '02983778', '02984767', '02985178', '02985499', '02985756', '02986386', '02987347', '02987755', '02989067', '02989262', '02989413', '02992500', '02992845', '02993614', '02994083', '02995329', '02995446', '02996340', '02996393', '02997017', '02998181', '02999637', '02999853']

pd.Series(folders).to_csv("valid_folders.txt", index=False, header=False)

In [ ]:
import base64, zlib
from pyspark.sql import functions as F

raw_encoded = """
...
"""
encoded = "e" + "".join(raw_encoded.split())

csv_text = zlib.decompress(base64.b64decode(encoded)).decode("utf-8")
with open("joined_df.csv", "w", encoding="utf-8") as f:
    f.write(csv_text)

joined_df = (
    spark.read.csv("joined_df.csv", header=True, escape='"')
    .withColumn("dog_list", F.from_json("dog_list", "array<string>"))
)

print("Row count:", joined_df.count())
joined_df.printSchema()

Row count: 19956
root
 |-- folder: string (nullable = true)
 |-- filename: string (nullable = true)
 |-- width: string (nullable = true)
 |-- height: string (nullable = true)
 |-- dog_list: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [6]:
import base64, zlib

encoded = "".join(raw_encoded.split())
print("Length:", len(encoded), "Mod 4:", len(encoded) % 4)
print("Starts with:", encoded[:15])
print("Ends with:", encoded[-15:])

Length: 479179 Mod 4: 3
Starts with: JyUvV2PJEeSJPh+
Ends with: drZZWn8/6RI3GI=
